<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [ ]:
import io
import math
import time
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
import cv2
from torchvision import transforms, models
from torchvision.ops import (
    nms, box_iou, distance_box_iou_loss,
)
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch.transforms import ToTensorV2
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [ ]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [ ]:
class HaloDataset(Dataset):
    """Halo Infinite детекция. Возвращает изображения и target с ббоксами в формате XYXY."""
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images  = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """
        Возвращаем:
            image: torch.Tensor (3, H, W)
            target: {
                'image_id': int,
                'boxes':  torch.Tensor (N, 4) в формате XYXY,
                'labels': torch.Tensor (N,) int64, классы с нуля
            }
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"])).convert("RGB")
        image = np.array(image)

        labels = [row["category"]] if isinstance(row["category"], int) else list(row['category'])
        labels = [int(l) - 1 for l in labels]            # классы с 0
        boxes_coco = [list(b) for b in row['bbox'].tolist()]   # формат: [x, y, w, h]

        # Albumentations принимает coco - отдадим как есть, конвертацию делаем после.
        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes_coco, labels=labels)
            image       = transformed["image"]
            boxes_coco  = transformed["bboxes"]
            labels      = transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        # coco (x, y, w, h)  -->  xyxy
        boxes_xyxy = []
        for b in boxes_coco:
            x, y, w, h = b
            boxes_xyxy.append([x, y, x + w, y + h])

        if len(boxes_xyxy) == 0:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes_t  = torch.tensor(boxes_xyxy, dtype=torch.float32)
            labels_t = torch.tensor(labels,     dtype=torch.int64)

        target = {
            "image_id": int(row["image_id"]),
            "boxes":    boxes_t,
            "labels":   labels_t,
        }
        return image, target


def collate_fn(batch):
    batch  = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]


Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [ ]:
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

IMG_SIZE = 640   # сторона квадратного входа, делится на максимальный stride 32

train_transform = A.Compose(
    [
        # --- геометрия: приводим к фикс. размеру + случайный кроп ---
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                      border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.8, 1.2), translate_percent=(-0.05, 0.05),
                 rotate=(-5, 5), p=0.5),

        # --- фотометрия ---
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10),
            A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10),
        ], p=0.5),

        # --- шум / дропаут ---
        A.OneOf([
            A.GaussNoise(var_limit=(5.0, 30.0)),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0),
        ], p=0.3),

        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # формат остаётся coco до конвертации в xyxy внутри Dataset
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3),
)

test_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                      border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3),
)


Не забываем инициализировать наш датасет

In [ ]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset  = HaloDataset(df_test,  transform=test_transform)

# Автоматически определяем число классов: смотрим max(category) во всех строках train
all_cats = set()
for cats in df_train["objects"].apply(lambda o: o["category"]):
    if isinstance(cats, (list, tuple, np.ndarray)):
        all_cats.update(int(c) for c in cats)
    else:
        all_cats.add(int(cats))
NUM_CLASSES = max(all_cats)        # классы в df идут с 1, после смещения [0, NUM_CLASSES-1]
print(f"NUM_CLASSES = {NUM_CLASSES}  (классы: {sorted(all_cats)})")

BATCH_SIZE  = 16
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS)
print(f"train batches: {len(train_loader)}, test batches: {len(test_loader)}")


## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [ ]:
class Backbone(nn.Module):
    """
    ResNet-based backbone, возвращает три feature map'a разных уровней (C3, C4, C5)
    со страйдами 8, 16, 32 относительно входа.

    Args:
        name:          'resnet18' | 'resnet34' | 'resnet50'
        pretrained:    использовать ли веса ImageNet
        unfreeze_last: сколько последних блоков (stem, layer1, layer2, layer3, layer4)
                       разморозить. 0 - всё заморожено, 5 - всё обучается.
    """
    def __init__(self, name="resnet50", pretrained=True, unfreeze_last=2):
        super().__init__()
        if name == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            net = models.resnet50(weights=weights)
            self.out_channels = [512, 1024, 2048]
        elif name == "resnet34":
            weights = models.ResNet34_Weights.DEFAULT if pretrained else None
            net = models.resnet34(weights=weights)
            self.out_channels = [128, 256, 512]
        elif name == "resnet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            net = models.resnet18(weights=weights)
            self.out_channels = [128, 256, 512]
        else:
            raise ValueError(f"Unknown backbone: {name}")

        self.stem   = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1 = net.layer1
        self.layer2 = net.layer2     # -> C3 (stride 8)
        self.layer3 = net.layer3     # -> C4 (stride 16)
        self.layer4 = net.layer4     # -> C5 (stride 32)

        # 1) заморозить всё
        for p in self.parameters():
            p.requires_grad = False
        # 2) разморозить последние k блоков
        blocks = [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]
        k = max(0, min(unfreeze_last, len(blocks)))
        for block in blocks[len(blocks) - k:]:
            for p in block.parameters():
                p.requires_grad = True

        # переводим BN в eval-режим для замороженных блоков (важно при unfreeze_last < 5)
        self._unfreeze_last = k
        self._set_bn_eval_for_frozen()

    def _set_bn_eval_for_frozen(self):
        blocks = [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]
        frozen = blocks[: len(blocks) - self._unfreeze_last]
        for block in frozen:
            for m in block.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()

    def train(self, mode=True):
        super().train(mode)
        # BN в замороженных блоках держим в eval, чтобы статистики ImageNet не портились
        self._set_bn_eval_for_frozen()
        return self

    def forward(self, x):
        x  = self.stem(x)
        x  = self.layer1(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return [c3, c4, c5]


### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [ ]:
class Neck(nn.Module):
    """
    Feature Pyramid Network (FPN). Получает [C3, C4, C5] и возвращает [P3, P4, P5]
    - все по out_channels каналов и со страйдами 8/16/32 соответственно.

    Шаги:
        1) lateral 1x1 conv на каждом уровне -> приводим число каналов к out_channels;
        2) top-down: апсемплим nearest-neighbour и складываем со скип-коннекшеном;
        3) сглаживающая 3x3 conv на каждом выходе (убирает aliasing от nearest).
    """
    def __init__(self, in_channels=(512, 1024, 2048), out_channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(c, out_channels, kernel_size=1) for c in in_channels
        ])
        self.smooth_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in in_channels
        ])
        self.out_channels = out_channels
        self.num_levels   = len(in_channels)

        # инициализация
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, features):
        # features: [C3, C4, C5]
        # 1) приводим к одинаковому числу каналов
        laterals = [lat_conv(f) for lat_conv, f in zip(self.lateral_convs, features)]

        # 2) top-down - идём с самого глубокого (P5 = lateral C5)
        for i in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(laterals[i],
                                      size=laterals[i - 1].shape[-2:],
                                      mode="nearest")
            laterals[i - 1] = laterals[i - 1] + upsampled

        # 3) сглаживание
        outs = [smooth(l) for smooth, l in zip(self.smooth_convs, laterals)]
        return outs   # [P3, P4, P5]


### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [ ]:
class Head(nn.Module):
    """
    Decoupled Head в стиле YOLOX. На каждом FPN-уровне:

        feature
          |-> [conv 3x3 + ReLU] x 2  -> cls_logits  (num_anchors * num_classes, H, W)
          |-> [conv 3x3 + ReLU] x 2  -> reg_deltas  (num_anchors * 4,           H, W)
                                     -> obj_logits  (num_anchors,               H, W)

    Веса между уровнями НЕ шарятся (в YOLOX тоже не шарятся; шеринг даёт чуть хуже mAP).
    """
    def __init__(self, in_channels=256, num_classes=2, num_anchors=3, num_levels=3,
                 stem_channels=256):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors
        self.num_levels  = num_levels

        self.cls_branches = nn.ModuleList()
        self.reg_branches = nn.ModuleList()
        self.cls_preds    = nn.ModuleList()
        self.reg_preds    = nn.ModuleList()
        self.obj_preds    = nn.ModuleList()

        for _ in range(num_levels):
            self.cls_branches.append(self._make_branch(in_channels, stem_channels))
            self.reg_branches.append(self._make_branch(in_channels, stem_channels))
            self.cls_preds.append(nn.Conv2d(stem_channels, num_anchors * num_classes, 1))
            self.reg_preds.append(nn.Conv2d(stem_channels, num_anchors * 4,           1))
            self.obj_preds.append(nn.Conv2d(stem_channels, num_anchors * 1,           1))

        self._init_weights()

    @staticmethod
    def _make_branch(in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.GroupNorm(32, out_ch), nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(32, out_ch), nn.SiLU(inplace=True),
        )

    def _init_weights(self):
        # focal-init для cls/obj - выход около prior=0.01,
        # это сильно стабилизирует начало обучения детектора с большим числом якорей
        prior = 0.01
        bias_init = -math.log((1 - prior) / prior)
        for m in [*self.cls_preds, *self.obj_preds]:
            nn.init.normal_(m.weight, std=0.01)
            nn.init.constant_(m.bias, bias_init)
        for m in self.reg_preds:
            nn.init.normal_(m.weight, std=0.01)
            nn.init.zeros_(m.bias)

    def forward(self, features):
        cls_outs, reg_outs, obj_outs = [], [], []
        for i, f in enumerate(features):
            cls_feat = self.cls_branches[i](f)
            reg_feat = self.reg_branches[i](f)
            cls_outs.append(self.cls_preds[i](cls_feat))
            reg_outs.append(self.reg_preds[i](reg_feat))
            # obj сидит на reg-ветке - как и в YOLOX
            obj_outs.append(self.obj_preds[i](reg_feat))
        return cls_outs, reg_outs, obj_outs


Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [ ]:
# ---------- AnchorGenerator ----------
class AnchorGenerator:
    """
    Генерирует якоря для каждого уровня FPN. На уровне со страйдом s якоря
    раскидываются в центрах ячеек stride/2 + i*s по обеим осям.

    Для каждой ячейки создаётся len(aspect_ratios) * len(scales) якорей вокруг центра.
    Базовый размер задан в anchor_sizes (по одному на уровень).
    """
    def __init__(self,
                 strides=(8, 16, 32),
                 anchor_sizes=(32, 64, 128),
                 aspect_ratios=(0.5, 1.0, 2.0),
                 scales=(1.0,)):
        self.strides       = list(strides)
        self.anchor_sizes  = list(anchor_sizes)
        self.aspect_ratios = list(aspect_ratios)
        self.scales        = list(scales)
        self.num_anchors_per_loc = len(self.aspect_ratios) * len(self.scales)

    def _base_anchors(self, base_size, device):
        anchors = []
        for ratio in self.aspect_ratios:
            for scale in self.scales:
                w = base_size * scale * math.sqrt(1.0 / ratio)
                h = base_size * scale * math.sqrt(ratio)
                anchors.append([-w / 2, -h / 2, w / 2, h / 2])
        return torch.tensor(anchors, dtype=torch.float32, device=device)   # (A, 4)

    @torch.no_grad()
    def generate(self, feature_sizes, device="cpu"):
        """Возвращает список тензоров [(H_l * W_l * A, 4)] для каждого уровня."""
        all_anchors = []
        for level, (H, W) in enumerate(feature_sizes):
            stride    = self.strides[level]
            base_size = self.anchor_sizes[level]
            base      = self._base_anchors(base_size, device)  # (A, 4)

            sx = (torch.arange(W, device=device, dtype=torch.float32) + 0.5) * stride
            sy = (torch.arange(H, device=device, dtype=torch.float32) + 0.5) * stride
            sy, sx = torch.meshgrid(sy, sx, indexing="ij")
            shifts = torch.stack([sx, sy, sx, sy], dim=-1).reshape(-1, 4)   # (HW, 4)

            anchors = base[None, :, :] + shifts[:, None, :]                 # (HW, A, 4)
            all_anchors.append(anchors.reshape(-1, 4))
        return all_anchors


def decode_boxes(anchors, deltas):
    """
    Стандартное декодирование (R-CNN-style):
        dx, dy = относительный сдвиг центра в долях ширины/высоты якоря,
        dw, dh = log-масштаб для ширины/высоты.

    anchors: (..., 4) xyxy,  deltas: (..., 4)  ->  (..., 4) xyxy.
    """
    aw  = anchors[..., 2] - anchors[..., 0]
    ah  = anchors[..., 3] - anchors[..., 1]
    acx = (anchors[..., 0] + anchors[..., 2]) * 0.5
    acy = (anchors[..., 1] + anchors[..., 3]) * 0.5

    dx, dy, dw, dh = deltas.unbind(-1)
    dw = torch.clamp(dw, max=4.0)        # |log(scale)| <= 4   => scale <= ~55
    dh = torch.clamp(dh, max=4.0)

    pcx = acx + dx * aw
    pcy = acy + dy * ah
    pw  = aw * torch.exp(dw)
    ph  = ah * torch.exp(dh)

    return torch.stack([
        pcx - pw * 0.5, pcy - ph * 0.5,
        pcx + pw * 0.5, pcy + ph * 0.5,
    ], dim=-1)


# ---------- Detector ----------
class Detector(nn.Module):
    def __init__(self,
                 num_classes,
                 backbone_name="resnet50",
                 unfreeze_last=2,
                 fpn_channels=256,
                 strides=(8, 16, 32),
                 anchor_sizes=(32, 64, 128),
                 aspect_ratios=(0.5, 1.0, 2.0)):
        super().__init__()
        self.num_classes = num_classes

        self.backbone = Backbone(backbone_name, pretrained=True, unfreeze_last=unfreeze_last)
        self.neck     = Neck(in_channels=tuple(self.backbone.out_channels),
                             out_channels=fpn_channels)
        self.anchor_generator = AnchorGenerator(
            strides=strides, anchor_sizes=anchor_sizes, aspect_ratios=aspect_ratios,
        )
        self.head = Head(in_channels=fpn_channels,
                         num_classes=num_classes,
                         num_anchors=self.anchor_generator.num_anchors_per_loc,
                         num_levels=self.neck.num_levels)
        self.num_anchors = self.anchor_generator.num_anchors_per_loc
        self.num_levels  = self.neck.num_levels

    def forward(self, x):
        feats = self.backbone(x)
        feats = self.neck(feats)
        cls_outs, reg_outs, obj_outs = self.head(feats)

        B = x.shape[0]
        feat_sizes      = [tuple(f.shape[-2:]) for f in feats]
        anchors_per_lvl = self.anchor_generator.generate(feat_sizes, device=x.device)
        anchors         = torch.cat(anchors_per_lvl, dim=0)   # (N_total, 4)

        cls_flat, reg_flat, obj_flat = [], [], []
        for c, r, o in zip(cls_outs, reg_outs, obj_outs):
            H, W = c.shape[-2:]
            cls_flat.append(
                c.permute(0, 2, 3, 1).reshape(B, H * W * self.num_anchors, self.num_classes))
            reg_flat.append(
                r.permute(0, 2, 3, 1).reshape(B, H * W * self.num_anchors, 4))
            obj_flat.append(
                o.permute(0, 2, 3, 1).reshape(B, H * W * self.num_anchors, 1))

        return {
            "cls":     torch.cat(cls_flat, dim=1),   # (B, N, C)  логиты
            "reg":     torch.cat(reg_flat, dim=1),   # (B, N, 4)  дельты
            "obj":     torch.cat(obj_flat, dim=1),   # (B, N, 1)  логит "есть объект"
            "anchors": anchors,                      # (N, 4) xyxy
        }


## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ - classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ - IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ - нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** - выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [ ]:
@torch.no_grad()
def simple_assigner(anchors, predictions, gt_boxes, gt_labels, num_classes,
                    pos_iou_thr=0.5, neg_iou_thr=0.4):
    """
    Простой max-IoU ассайнер (как в Faster R-CNN). Используем как baseline и для warm-up.
    Возвращает (target_labels, target_boxes, pos_mask, norm_weight).
    """
    device = anchors.device
    N = anchors.shape[0]
    M = gt_boxes.shape[0]

    target_labels = torch.full((N,), num_classes, dtype=torch.long,   device=device)
    target_boxes  = torch.zeros((N, 4),            dtype=torch.float32, device=device)
    pos_mask      = torch.zeros(N,                 dtype=torch.bool,    device=device)
    norm_weight   = torch.zeros(N,                 dtype=torch.float32, device=device)

    if M == 0:
        return target_labels, target_boxes, pos_mask, norm_weight

    iou = box_iou(anchors, gt_boxes)              # (N, M)
    max_iou, max_iou_idx = iou.max(dim=1)         # (N,)

    pos = max_iou > pos_iou_thr
    target_labels[pos] = gt_labels[max_iou_idx[pos]]
    target_boxes[pos]  = gt_boxes[max_iou_idx[pos]]
    pos_mask[pos]      = True

    # гарантируем хотя бы один якорь на GT
    _, best_anchor_for_gt = iou.max(dim=0)        # (M,)
    target_labels[best_anchor_for_gt] = gt_labels
    target_boxes[best_anchor_for_gt]  = gt_boxes
    pos_mask[best_anchor_for_gt]      = True

    norm_weight[pos_mask] = max_iou[pos_mask].clamp(min=0.0)
    return target_labels, target_boxes, pos_mask, norm_weight


@torch.no_grad()
def TAL_assigner(anchors, predictions, gt_boxes, gt_labels, num_classes,
                 topk=13, alpha=1.0, beta=6.0, eps=1e-9):
    """
    Task Alignment Learning (TOOD, sec. 3.2).
    Шаги:
        1) t = s^alpha * u^beta, где s - cls-score для GT-класса, u - IoU(pred, GT);
        2) фильтрация по condition «центр якоря внутри GT»;
        3) top-k метрик на каждый GT;
        4) если якорь предложен сразу нескольким GT - оставляем GT с макс. IoU;
        5) дополнительно возвращаем нормированную метрику для soft-label на cls
           (стандартный трюк из статьи).

    predictions: dict с двумя ключами:
        'cls'            (N, C) - сигмоид cls-логитов (вероятности),
        'decoded_boxes'  (N, 4) - XYXY-боксы, декодированные из дельт.
    """
    device = anchors.device
    N = anchors.shape[0]
    M = gt_boxes.shape[0]

    target_labels = torch.full((N,), num_classes, dtype=torch.long,   device=device)
    target_boxes  = torch.zeros((N, 4),            dtype=torch.float32, device=device)
    pos_mask      = torch.zeros(N,                 dtype=torch.bool,    device=device)
    norm_weight   = torch.zeros(N,                 dtype=torch.float32, device=device)

    if M == 0:
        return target_labels, target_boxes, pos_mask, norm_weight

    pred_cls   = predictions["cls"]              # (N, C)
    pred_boxes = predictions["decoded_boxes"]    # (N, 4)

    # --- 1) alignment metric t = s^alpha * u^beta ---
    iou = box_iou(pred_boxes, gt_boxes)          # (N, M)
    s   = pred_cls[:, gt_labels]                 # (N, M) - score за класс соотв. GT
    t   = s.clamp(min=eps).pow(alpha) * iou.clamp(min=eps).pow(beta)   # (N, M)

    # --- 2) центр якоря внутри GT ---
    acx = (anchors[:, 0] + anchors[:, 2]) * 0.5
    acy = (anchors[:, 1] + anchors[:, 3]) * 0.5
    inside = (
        (acx[:, None] >= gt_boxes[None, :, 0]) &
        (acx[:, None] <= gt_boxes[None, :, 2]) &
        (acy[:, None] >= gt_boxes[None, :, 1]) &
        (acy[:, None] <= gt_boxes[None, :, 3])
    )
    t = t * inside.float()

    # --- 3) top-k якорей на каждый GT (только с положительной alignment-метрикой) ---
    k_eff = min(topk, N)
    topk_vals, topk_idx = t.topk(k_eff, dim=0)               # (k, M)
    valid_mask = topk_vals > 0                                # (k, M)
    cand = torch.zeros_like(t)                                # (N, M)
    for j in range(M):
        sel = topk_idx[:, j][valid_mask[:, j]]
        cand[sel, j] = 1.0

    # --- 4) разрешаем коллизии: один якорь -> max-IoU GT ---
    multi = cand.sum(dim=1) > 1
    if multi.any():
        best_j = iou[multi].argmax(dim=1)
        cand[multi] = 0.0
        cand[multi, best_j] = 1.0

    positives    = cand.sum(dim=1) > 0
    assigned_gt  = cand.argmax(dim=1)            # (N,) валидно только в positives

    target_labels[positives] = gt_labels[assigned_gt[positives]]
    target_boxes[positives]  = gt_boxes[assigned_gt[positives]]
    pos_mask[positives]      = True

    # --- 5) нормированная метрика для soft-cls-таргета (TOOD eq.7) ---
    # t_norm = t * (max_iou_per_gt / max_t_per_gt)
    t_pos       = t * cand                                    # (N, M) - только выбранные якоря
    max_t_per_gt   = t_pos.amax(dim=0, keepdim=True).clamp(min=eps)
    max_iou_per_gt = (iou * cand).amax(dim=0, keepdim=True)
    t_norm = t_pos * (max_iou_per_gt / max_t_per_gt)
    norm_weight[positives] = t_norm[positives, assigned_gt[positives]]

    return target_labels, target_boxes, pos_mask, norm_weight


### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [ ]:
from torchvision.ops import distance_box_iou_loss

In [ ]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [ ]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [ ]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

In [ ]:
def diou_loss(pred_boxes, gt_boxes, reduction="mean", eps=1e-7):
    """
    Distance-IoU loss.  L = 1 - IoU + d^2 / c^2

    pred_boxes, gt_boxes - (..., 4) в формате XYXY.

    Реализация полностью совпадает с torchvision.ops.distance_box_iou_loss
    (то же eps в IoU и тот же eps в знаменателе диагонали).
    """
    pred_boxes = pred_boxes.float()
    gt_boxes   = gt_boxes.float()

    px1, py1, px2, py2 = pred_boxes.unbind(-1)
    gx1, gy1, gx2, gy2 = gt_boxes.unbind(-1)

    # --- 1) IoU ---
    pred_area = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    gt_area   = (gx2 - gx1).clamp(min=0) * (gy2 - gy1).clamp(min=0)

    ix1 = torch.maximum(px1, gx1); iy1 = torch.maximum(py1, gy1)
    ix2 = torch.minimum(px2, gx2); iy2 = torch.minimum(py2, gy2)
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)
    union = pred_area + gt_area - inter
    iou   = inter / (union + eps)

    # --- 2) диагональ выпуклой оболочки ---
    cx1 = torch.minimum(px1, gx1); cy1 = torch.minimum(py1, gy1)
    cx2 = torch.maximum(px2, gx2); cy2 = torch.maximum(py2, gy2)
    c2  = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2 + eps

    # --- 3) расстояние между центрами ---
    pcx = (px1 + px2) * 0.5; pcy = (py1 + py2) * 0.5
    gcx = (gx1 + gx2) * 0.5; gcy = (gy1 + gy2) * 0.5
    d2  = (pcx - gcx) ** 2 + (pcy - gcy) ** 2

    loss = 1.0 - iou + d2 / c2

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


In [ ]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [ ]:
# ============================================================
#   Loss, фильтрация предсказаний, Runner, тренировка
# ============================================================

class ComputeLoss(nn.Module):
    """
    Лосс с тремя слагаемыми:
        cls - BCE-with-logits (multi-label, по числу классов),
        obj - BCE-with-logits  для objectness,
        box - DIoU на положительных якорях.

    assigner - функция вида simple_assigner / TAL_assigner.
    """
    def __init__(self, num_classes, assigner,
                 lambda_cls=1.0, lambda_obj=1.0, lambda_box=2.0,
                 use_soft_cls=True):
        super().__init__()
        self.num_classes  = num_classes
        self.assigner     = assigner
        self.lambda_cls   = lambda_cls
        self.lambda_obj   = lambda_obj
        self.lambda_box   = lambda_box
        self.use_soft_cls = use_soft_cls

    def forward(self, outputs, targets):
        cls_logits = outputs["cls"]      # (B, N, C)
        reg_deltas = outputs["reg"]      # (B, N, 4)
        obj_logits = outputs["obj"]      # (B, N, 1)
        anchors    = outputs["anchors"]  # (N, 4)
        device     = cls_logits.device
        B, N, C    = cls_logits.shape

        total_cls = torch.zeros((), device=device)
        total_obj = torch.zeros((), device=device)
        total_box = torch.zeros((), device=device)
        total_pos = 0

        for b in range(B):
            gt_boxes  = targets[b]["boxes"].to(device)
            gt_labels = targets[b]["labels"].to(device)

            with torch.no_grad():
                decoded   = decode_boxes(anchors, reg_deltas[b].detach())
                cls_probs = cls_logits[b].detach().sigmoid()

            tgt_labels, tgt_boxes, pos_mask, norm_w = self.assigner(
                anchors,
                {"decoded_boxes": decoded, "cls": cls_probs},
                gt_boxes, gt_labels,
                num_classes=self.num_classes,
            )
            n_pos = int(pos_mask.sum().item())
            total_pos += n_pos

            # --- cls (one-hot или soft) ---
            cls_target = torch.zeros_like(cls_logits[b])
            if n_pos > 0:
                if self.use_soft_cls:
                    cls_target[pos_mask, tgt_labels[pos_mask]] = norm_w[pos_mask]
                else:
                    cls_target[pos_mask, tgt_labels[pos_mask]] = 1.0
            total_cls = total_cls + F.binary_cross_entropy_with_logits(
                cls_logits[b], cls_target, reduction="sum")

            # --- obj ---
            obj_target = pos_mask.float().unsqueeze(-1)
            total_obj  = total_obj + F.binary_cross_entropy_with_logits(
                obj_logits[b], obj_target, reduction="sum")

            # --- box (DIoU) на позитивах ---
            if n_pos > 0:
                pred_boxes_pos = decode_boxes(anchors[pos_mask], reg_deltas[b][pos_mask])
                gt_pos         = tgt_boxes[pos_mask]
                # взвешиваем по нормированной alignment-метрике (как в TOOD)
                w   = norm_w[pos_mask].clamp(min=1e-6) if self.use_soft_cls else None
                per = diou_loss(pred_boxes_pos, gt_pos, reduction="none")
                if w is not None:
                    total_box = total_box + (per * w).sum() / w.sum().clamp(min=1.0)
                    total_box = total_box * n_pos     # для корректного нормирования ниже
                else:
                    total_box = total_box + per.sum()

        norm_pos = max(total_pos, 1)
        loss_cls = total_cls / norm_pos * self.lambda_cls
        loss_obj = total_obj / (B * N)              * self.lambda_obj
        loss_box = total_box / norm_pos              * self.lambda_box

        return {
            "loss":     loss_cls + loss_obj + loss_box,
            "loss_cls": loss_cls.detach(),
            "loss_obj": loss_obj.detach(),
            "loss_box": loss_box.detach(),
            "n_pos":    norm_pos,
        }


def filter_predictions(outputs, score_threshold=0.05, nms_threshold=0.5,
                       max_dets=300, num_classes=None):
    """
    Из сырых выходов Detector делает список dict-ов с финальными детекциями
    в формате torchmetrics: {'boxes' (xyxy), 'scores', 'labels'}.
    """
    cls_logits = outputs["cls"]
    reg_deltas = outputs["reg"]
    obj_logits = outputs["obj"]
    anchors    = outputs["anchors"]
    B, N, C    = cls_logits.shape
    if num_classes is None:
        num_classes = C

    cls_scores = cls_logits.sigmoid()
    obj_scores = obj_logits.sigmoid().squeeze(-1)        # (B, N)

    results = []
    for b in range(B):
        boxes_b = decode_boxes(anchors, reg_deltas[b])   # (N, 4)
        scores_b = cls_scores[b] * obj_scores[b].unsqueeze(-1)   # (N, C)

        all_boxes, all_scores, all_labels = [], [], []
        for c in range(num_classes):
            sc = scores_b[:, c]
            keep_thr = sc > score_threshold
            if keep_thr.sum() == 0:
                continue
            bx = boxes_b[keep_thr]
            sc = sc[keep_thr]
            keep = nms(bx, sc, nms_threshold)
            all_boxes.append(bx[keep])
            all_scores.append(sc[keep])
            all_labels.append(torch.full((keep.numel(),), c,
                                         dtype=torch.int64, device=bx.device))

        if all_boxes:
            boxes  = torch.cat(all_boxes)
            scores = torch.cat(all_scores)
            labels = torch.cat(all_labels)
            if boxes.shape[0] > max_dets:
                topk = scores.topk(max_dets).indices
                boxes, scores, labels = boxes[topk], scores[topk], labels[topk]
        else:
            boxes  = torch.zeros((0, 4),   device=cls_logits.device)
            scores = torch.zeros((0,),     device=cls_logits.device)
            labels = torch.zeros((0,), dtype=torch.int64, device=cls_logits.device)

        results.append({"boxes": boxes, "scores": scores, "labels": labels})
    return results


class Runner:
    """Минимальный обучатель, как в семинаре. Поддерживает смену assigner'а по эпохе."""
    def __init__(self, model, optimizer, loss_fn,
                 assigner_warmup=None, warmup_epochs=0,
                 scheduler=None, device=DEVICE, grad_clip=10.0):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.assigner_main   = loss_fn.assigner
        self.assigner_warmup = assigner_warmup
        self.warmup_epochs   = warmup_epochs
        self.scheduler = scheduler
        self.device = device
        self.grad_clip = grad_clip
        self.history = defaultdict(list)

    def _set_assigner_for_epoch(self, epoch):
        if self.assigner_warmup is not None and epoch < self.warmup_epochs:
            self.loss_fn.assigner = self.assigner_warmup
        else:
            self.loss_fn.assigner = self.assigner_main

    def _run_train_epoch(self, loader, epoch):
        self.model.train()
        self._set_assigner_for_epoch(epoch)

        running = defaultdict(float)
        pbar = tqdm(loader, desc=f"train {epoch}", leave=False)
        for images, targets in pbar:
            images = images.to(self.device, non_blocking=True)
            outputs = self.model(images)
            losses  = self.loss_fn(outputs, targets)

            self.optimizer.zero_grad(set_to_none=True)
            losses["loss"].backward()
            if self.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
            if self.scheduler is not None:
                self.scheduler.step()

            for k, v in losses.items():
                if k == "n_pos":
                    continue
                running[k] += float(v.detach())
            pbar.set_postfix(loss=f"{float(losses['loss']):.3f}",
                             n_pos=losses['n_pos'])

        n = len(loader)
        for k in running:
            running[k] /= max(n, 1)
            self.history[k].append(running[k])
        return dict(running)

    @torch.no_grad()
    def validate(self, loader, filter_predictions_func=filter_predictions,
                 score_threshold=0.05, nms_threshold=0.5, max_dets=300):
        self.model.eval()
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
        for images, targets in tqdm(loader, desc="validate", leave=False):
            images = images.to(self.device, non_blocking=True)
            outputs = self.model(images)
            preds = filter_predictions_func(
                outputs, score_threshold=score_threshold,
                nms_threshold=nms_threshold, max_dets=max_dets,
                num_classes=self.model.num_classes,
            )
            # перевод на cpu для torchmetrics
            preds_cpu   = [{k: v.detach().cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{k: (v.detach().cpu() if torch.is_tensor(v) else v)
                            for k, v in t.items()} for t in targets]
            metric.update(preds_cpu, targets_cpu)
        return metric.compute()

    def fit(self, train_loader, val_loader, epochs):
        best_map = 0.0
        for epoch in range(epochs):
            t0 = time.time()
            tr = self._run_train_epoch(train_loader, epoch)
            val = self.validate(val_loader)
            map_ = float(val["map"])
            map50 = float(val["map_50"])
            self.history["val_map"].append(map_)
            self.history["val_map50"].append(map50)
            if map_ > best_map:
                best_map = map_
            print(f"epoch {epoch:02d}  loss={tr['loss']:.3f}  "
                  f"mAP={map_:.4f}  mAP50={map50:.4f}  "
                  f"dt={time.time()-t0:.0f}s  best mAP={best_map:.4f}")
        return self.history


# ============================================================
#   Конфиг и обучение
# ============================================================

EPOCHS         = 25
WARMUP_EPOCHS  = 5            # первые эпохи учим простым ассайнером, потом - TAL
LR             = 5e-4
WEIGHT_DECAY   = 1e-4

model = Detector(
    num_classes=NUM_CLASSES,
    backbone_name="resnet50",
    unfreeze_last=3,           # размораживаем layer2/3/4
    fpn_channels=256,
    strides=(8, 16, 32),
    anchor_sizes=(32, 64, 128),
    aspect_ratios=(0.5, 1.0, 2.0),
).to(DEVICE)

# Только обучаемые параметры -> оптимизатор
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable) / 1e6:.2f}M")

optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=EPOCHS * max(len(train_loader), 1),
    pct_start=0.1, anneal_strategy="cos",
)

loss_fn = ComputeLoss(
    num_classes=NUM_CLASSES,
    assigner=TAL_assigner,        # основной ассайнер - TAL
    lambda_cls=1.0, lambda_obj=1.0, lambda_box=2.0,
    use_soft_cls=True,
)

runner = Runner(
    model, optimizer, loss_fn,
    assigner_warmup=simple_assigner,    # warm-up - простой IoU-ассайнер (TIP №3)
    warmup_epochs=WARMUP_EPOCHS,
    scheduler=scheduler,
    device=DEVICE,
)

history = runner.fit(train_loader, test_loader, EPOCHS)


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(model, dataloader, filter_predictions_func,
             box_format="xyxy", device=DEVICE,
             score_threshold=0.05, nms_threshold=0.5, max_dets=300, **kwargs):
    """Внешняя обёртка над валидацией: вернёт COCO-mAP (0.5:0.95)."""
    model.eval()
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)
        preds = filter_predictions_func(
            outputs,
            score_threshold=score_threshold,
            nms_threshold=nms_threshold,
            max_dets=max_dets,
            num_classes=model.num_classes,
            **kwargs,
        )
        preds_cpu   = [{k: v.detach().cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: (v.detach().cpu() if torch.is_tensor(v) else v)
                        for k, v in t.items()} for t in targets]
        metric.update(preds_cpu, targets_cpu)
    out = metric.compute()
    print({k: float(v) for k, v in out.items() if torch.is_tensor(v) and v.numel() == 1})
    return out["map"].item()


# Финальная метрика на тесте
final_map = validate(model, test_loader, filter_predictions)
print(f"Final test mAP@[0.5:0.95] = {final_map:.4f}")


## Ответы на вопросы

### 1. Какой метод label assignment помогает лучше обучаться модели? Почему?

В моих экспериментах **TAL (Task Alignment Learning)** даёт заметно более высокий
mAP, чем простой max-IoU ассайнер. Причины:

- **Совместное согласование двух задач.** Простой IoU-ассайнер выбирает позитивы
  только по геометрии (IoU якоря с GT) и совершенно не учитывает classification
  score. Из-за этого один и тот же якорь может быть «отличным» по локализации, но
  модель уверенно классифицирует его как фон - и обратное. TAL вводит метрику
  $t = s^\alpha \cdot u^\beta$, которая одновременно требует и высокий
  cls-score, и высокий IoU; позитивами становятся якоря, где обе задачи
  «согласованы».
- **Динамические top-k вместо порога.** TAL берёт фиксированное число лучших
  якорей на каждый GT, поэтому количество позитивов не зависит от субъективного
  IoU-threshold'а и хорошо масштабируется на объекты разного размера.
- **Soft-label на cls.** Нормированная alignment-метрика используется как мягкий
  cls-таргет - модель «знает», насколько хорошо выровнено это предсказание, и
  получает более калиброванные скоры. На инференсе это даёт чистый ранжированный
  список ббоксов и меньше «битвы» NMS.
- **Учёт условия «центр внутри GT».** Это фильтрует абсурдные позитивы (якорь
  попал по IoU за счёт частичного перекрытия, но его центр вне GT) - особенно
  важно на мелких уровнях FPN.

Дополнительный приём, который помог стабилизировать обучение, - **warm-up на
простом IoU-ассайнере**: в первые 5 эпох модель ещё не предсказывает ничего
осмысленного, и в TAL метрика $s$ почти случайна, поэтому top-k выбирает плохие
позитивы и обучение долго раскачивается. После warm-up предсказания уже
информативны - переходим на TAL и получаем буст.

### 2. Какое из улучшений внесло наибольший вклад? Почему?

**Самый большой буст - FPN + multi-scale anchors.** Базовый детектор с одной
ветки сильно страдает на мелких объектах (Halo Infinite - это in-game-скриншоты
с фигурами разных размеров). FPN добавляет признаки трёх уровней (страйды
8/16/32), и каждый уровень специализируется на «своём» масштабе:

- P3 (stride 8) - мелкие игроки/головы вдали;
- P4 (stride 16) - средние объекты;
- P5 (stride 32) - крупные/близкие.

После FPN mAP на mAP@0.5 у меня поднимался очень заметно - крупный прирост
именно за счёт того, что мелкие GT-боксы наконец-то имеют якоря на feature-map
со страйдом 8 (на 32-страйде они физически не «ловятся»).

Второе по эффекту - переход к **DIoU/IoU-based loss** вместо SmoothL1: лосс
напрямую оптимизирует то, что считается метрикой (IoU). Косвенно это улучшает и
ранжирование по NMS - у предсказаний с высоким score обычно и высокий IoU с GT.

### 3. Какое улучшение вообще не сдвинуло метрику? Почему так?

На моих прогонах наименьший эффект дала **разморозка большего числа слоёв
бэкбона**. При переходе от `unfreeze_last=2` (только `layer4`) к
`unfreeze_last=4` (`layer1..layer4`) mAP практически не изменился - буквально в
шуме между запусками. Причина, на мой взгляд:

- Halo Infinite - это домен, относительно близкий к ImageNet (натуральные RGB
  изображения с прямоугольными объектами), поэтому низкоуровневые признаки
  (текстуры, простые контуры) переносятся хорошо и без дообучения.
- Высокоуровневое представление, которое нужно адаптировать, и так уже
  размораживается с `unfreeze_last=2`.
- Малый размер датасета: больше обучаемых параметров → выше шанс переобучения,
  это съедает потенциальный выигрыш.

То же касается *слишком агрессивных аугментаций* (сильный rotate / CoarseDropout
со 16 дырками): они либо не двигают метрику, либо ухудшают её, потому что
делают распределение train-картинок не похожим на test.

**Вывод по приоритетам улучшений:** FPN → IoU-loss → TAL → разморозка/аугментации.
Архитектурные изменения и качество label assignment почти всегда важнее, чем
тонкая настройка трейна.
